# Standalone Orbital Visualiser

### Calculation of Cubes

In [1]:
# Calculation Dependencies
from aiida.engine import run_get_node
from aiida.orm import FolderData

from topworkchain import PrototypeTopWorkChain
from aiida import load_profile

In [2]:
# Load AiiDA profile
load_profile()

Profile<uuid='81d56c42503449ecb75a3ffc3bcb5e45' name='presto'>

In [3]:
# Run the workchain that calculates NTOs, converts them to cube then compresses and returns them as FolderData
results, node = run_get_node(PrototypeTopWorkChain)

# Load the FolderData node containing the compressed cube files
cube_folder = results["cube_folder"]

# Load the Dict node containing all of the relevant details of the transitions.
transition_info_node = results["transition_info"]

# Convert to a standard Python dict.
transition_info = transition_info_node.get_dict()

04/25/2026 12:25:41 PM <82442> aiida.broker.rabbitmq: [WARNING] RabbitMQ v4.2.3 is not supported and will cause unexpected problems!
04/25/2026 12:25:41 PM <82442> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
04/25/2026 12:25:41 PM <82442> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.


uuid: 43e685d7-c978-47ee-99be-7d9a9fc19f58 (pk: 11165) (subworkchains.OrcaWorkChain)
uuid: 92562d06-360f-40be-989d-03b3fe16456f (pk: 11190)
uuid: 3bba8324-aaa5-4732-a89f-785b8974255c (pk: 11204)
uuid: 3fb1552b-5914-4820-b43d-cff6301e43a0 (pk: 11219)
uuid: 09ad1e79-f0de-4eff-b20a-8e55e0dba3e4 (pk: 11233)
uuid: 508b6d3d-a77f-4b36-8f4f-50f70271ad4b (pk: 11248)
uuid: ab8e8240-1556-4ac9-b918-059bd76de48f (pk: 11262)
uuid: 997ff807-dbf9-4a79-9522-a23033be9580 (pk: 11276)
uuid: 09f345f5-caec-4ab5-85e0-0a5cc5d456ba (pk: 11290)
uuid: 42aa8ccb-e13d-4ebc-b890-405b7bce23e7 (pk: 11305)
uuid: cc430170-00b6-4f1e-8417-2e47bd2c1ab6 (pk: 11319)
uuid: cbfdecd2-1561-4c5c-b360-fa2d2a196609 (pk: 11334)
uuid: 883d308f-8cc5-4c6e-84f5-f3a7dda84c1b (pk: 11348)
uuid: 30087012-a689-44f1-b32d-f34485138358 (pk: 11363)
uuid: 7fbb429e-f782-43f1-a2c9-6965441e004b (pk: 11377)
uuid: 89d88823-16bc-4ef7-a082-cb1e7962eef7 (pk: 11391)
uuid: c8e43f4c-e863-4acf-be29-b7f3e34b3e53 (pk: 11405)
uuid: f1f648be-67e8-4565-8159-a7e35

### Visualisation

In [4]:
import nglview as nv
import ipywidgets as widgets
from ipywidgets import Layout

import ase.io
from pathlib import Path

In [5]:
#----CUBE FILE HANDLING-----
# Check if local directory exists, if it does clear as cube files can be pretty big and there are a lot.
# If not directory, create one

output_dir = Path("extracted_cubes")

if output_dir.is_dir():
    for item in output_dir.iterdir():
        item.unlink()
else:
    output_dir.mkdir(exist_ok=True)

# Create a list of the cube files in the data node
names = cube_folder.list_object_names()

# Read in the cube files and write them to temporary files
for name in names:
    with cube_folder.open(name, mode="rb") as file_in:
        with open(f"{output_dir}/{name}.cube", "wb") as file_out:
            file_out.write(file_in.read())


# ----EXCITED STATE DROPDOWN----
specific_excitations = list(transition_info.keys())

# Create dropdown widgets for selecting the cube files
excitation_dropdown = widgets.Dropdown(
    options=specific_excitations,
    value=specific_excitations[0] if specific_excitations else None,
    description="Excitation (s)"
 )


# ----TRANSITION STATE DROPDOWN----
#Function to change the options available in the transition_dropdown.
def transition_update(change):
    #Isolate specific relevant transitions.
    specific_transitions = transition_info[excitation_dropdown.value]
    #Adjust format of transitions.
    specific_transitions2 = []
    for each_transition in specific_transitions:
        specific_transitions2.append((((each_transition[0])[0]+" -> "+(each_transition[0])[1]+" ("+each_transition[1]+")"), each_transition))
    transition_dropdown.options = specific_transitions2

transition_dropdown = widgets.Dropdown(
    description="Transition"
 )

#Run transition_update when excitation_dropdown is clicked.
excitation_dropdown.observe(transition_update, names="value")

#Initialise the transition_dropdown with initial options and value.
transition_update({"new": excitation_dropdown.value})
transition_dropdown.value = transition_dropdown.options[0][1] if transition_dropdown.options else None


# ----NGL VIEWER----
# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube"}"
CUBE_PATH_2 = f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube"}"

# Create NGL views for the selected cube files
molecule1 = nv.show_ase(ase.io.read(CUBE_PATH_1))
molecule2 = nv.show_ase(ase.io.read(CUBE_PATH_2))
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube")
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube")


# ----ISOVALUES AND COLORS----
# Create sliders for adjusting the isovalue (not visible in the NGL view, but used to set the isovalue of the surfaces)
positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
 )
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
 )

# List of colour blind friendly colours
colours = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

# To store colour buttons
pos_palette_buttons = []
neg_palette_buttons = []

def set_positive_colour(colour):
    selected_colours["positive"] = colour
    redraw_surfaces()

def set_negative_colour(colour):
    selected_colours["negative"] = colour
    redraw_surfaces()

# Create buttons for each colour for both palettes
for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_positive_colour(c))
    pos_palette_buttons.append(b)

for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_negative_colour(c))
    neg_palette_buttons.append(b)

pos_palette = widgets.HBox(pos_palette_buttons)
neg_palette = widgets.HBox(neg_palette_buttons)

# Preset colours 
selected_colours = {
    "positive": "#377eb8",
    "negative": "#e41a1c"
}

# Create a box to contain the colour palettes
palette_box = widgets.VBox([
    widgets.HTML("<b>Positive colour</b>"),
    pos_palette,
    widgets.HTML("<b>Negative colour</b>"),
    neg_palette
    
])

# Create drop down for displaying colours
accordion = widgets.Accordion(children=[palette_box])
accordion.set_title(0, 'Select colours')
accordion.selected_index = None  # start closed
accordion.layout = Layout(width="25%")

status = widgets.HTML()

# Define functions to redraw the surfaces when the sliders or colour scheme are changed
def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    #print("s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube")
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[0])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    #print("s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube")
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"."+((transition_dropdown.value[0])[1])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_2()

# Set up observers to redraw the surfaces when the sliders or colour scheme are changed
for control in [positive_level, negative_level]:
    control.observe(redraw_surfaces, names="value")

excitation_dropdown.observe(update_NTO_1, names="value")
transition_dropdown.observe(update_NTO_1, names="value")
excitation_dropdown.observe(update_NTO_2, names="value")
transition_dropdown.observe(update_NTO_2, names="value")

redraw_surfaces()

# Arrange the widgets in a layout
controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        accordion,
        status,
    ],
 )
molecule1_box = widgets.VBox([excitation_dropdown, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([transition_dropdown, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))
